The below code imports all necessary for cleaning up the data.

In [2]:
import pandas as pd
import os
from glob import glob
import gc
import numpy as np

Create defined code analyze_datasets, which starteds columns to keep. Then running a for loop we loop through each file, uploading, adjusting, and writing the adjustments to the file. This creates a new file 00-00-0000-analytics.parquet, which we can upload for querying in aws Athena.
    

In [3]:
def analyze_datasets(df):
    columns_to_keep = [
        'Timestamp',
        'Dst Port',
        'Protocol',
        'Flow Duration',
        'Flow Byts/s',
        'Flow Pkts/s',
        'Tot Fwd Pkts',
        'Tot Bwd Pkts',
        'Label'
    ]
    
    return df[columns_to_keep]

parquet_files = glob("cleaned/*.parquet")

for file in parquet_files:

    print(f"Loading {file}...")
    df = pd.read_parquet(file)

    print(f"Adjusting {file}...")
    analytics_df = analyze_datasets(df)

    output_name = os.path.basename(file).replace("-clean.parquet", "-analytics.parquet")
    output_path = f"analytics/{output_name}"
    
    print(f"Writing {file}...")
    analytics_df.to_parquet(output_path, index=False)


    print(f"Finished {file}")
    print("Original:", df.shape)
    print("Adjusted:", analytics_df.shape)
    print("Saved to:", output_path)
    print()

    del df
    del analytics_df
    gc.collect()
    
    

Loading cleaned/02-28-2018-clean.parquet...
Adjusting cleaned/02-28-2018-clean.parquet...
Writing cleaned/02-28-2018-clean.parquet...
Finished cleaned/02-28-2018-clean.parquet
Original: (604313, 80)
Adjusted: (604313, 9)
Saved to: analytics/02-28-2018-analytics.parquet

Loading cleaned/03-01-2018-clean.parquet...
Adjusting cleaned/03-01-2018-clean.parquet...
Writing cleaned/03-01-2018-clean.parquet...
Finished cleaned/03-01-2018-clean.parquet
Original: (329193, 80)
Adjusted: (329193, 9)
Saved to: analytics/03-01-2018-analytics.parquet

Loading cleaned/03-02-2018-clean.parquet...
Adjusting cleaned/03-02-2018-clean.parquet...
Writing cleaned/03-02-2018-clean.parquet...
Finished cleaned/03-02-2018-clean.parquet
Original: (1040558, 80)
Adjusted: (1040558, 9)
Saved to: analytics/03-02-2018-analytics.parquet

Loading cleaned/02-23-2018-clean.parquet...
Adjusting cleaned/02-23-2018-clean.parquet...
Writing cleaned/02-23-2018-clean.parquet...
Finished cleaned/02-23-2018-clean.parquet
Original:

Double check shape and make sure that all data was edited correctly. Using various tools like shape, describe, to see the changes that have taken affect. Then also run it against previous cleaned files to see changes.

In [4]:
df = pd.read_parquet("cleaned/02-20-2018-clean.parquet")
print(df.shape)

(7911980, 84)


In [6]:
df.shape

(7911980, 84)

In [7]:
analytics_df = df[
    [
        'Timestamp',
        'Src IP',
        'Src Port',
        'Dst IP',
        'Dst Port',
        'Protocol',
        'Flow Duration',
        'Flow Byts/s',
        'Flow Pkts/s',
        'Tot Fwd Pkts',
        'Tot Bwd Pkts',
        'Label'
    ]
    ]

In [8]:
print(analytics_df.shape)

(7911980, 12)


In [10]:
analytics_df.describe()

/home/tman/security-data-lake/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/tman/security-data-lake/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Timestamp,Src Port,Dst Port,Protocol,Flow Duration,Flow Byts/s,Flow Pkts/s,Tot Fwd Pkts,Tot Bwd Pkts
count,7911980,7.911980e+06,7.911980e+06,7.911980e+06,7.911980e+06,7.911980e+06,7.911980e+06,7.911980e+06,7.911980e+06
mean,2018-02-20 07:17:12.770044,4.548526e+04,7.156902e+03,9.378914e+00,1.359616e+07,inf,inf,1.705120e+01,6.744294e+00
min,2018-02-20 01:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.666667e-02,1.000000e+00,0.000000e+00
25%,2018-02-20 03:23:48,4.973900e+04,5.300000e+01,6.000000e+00,4.670000e+02,4.132418e+00,4.545068e+00,1.000000e+00,1.000000e+00
50%,2018-02-20 08:53:59,5.244400e+04,8.000000e+01,6.000000e+00,5.564750e+04,1.102433e+03,7.405213e+01,2.000000e+00,1.000000e+00
75%,2018-02-20 10:48:52,5.783500e+04,3.389000e+03,1.700000e+01,3.208822e+06,9.490334e+04,5.617978e+03,7.000000e+00,5.000000e+00
max,2018-02-20 12:59:59,6.553500e+04,6.553500e+04,1.700000e+01,1.200000e+08,inf,inf,2.800430e+05,2.580600e+04
std,NaN,2.094715e+04,1.704274e+04,5.254439e+00,3.249737e+07,NaN,NaN,1.247256e+03,1.584191e+02


In [9]:
analytics_df.to_parquet("analytics/02-20-2018-analytics-extra.parquet", index=False)

In [12]:
analytics_files = glob("analytics/*.parquet")

Check files shape. Compare against cleaned folder.

In [16]:
for file in analytics_files:
    df = pd.read_parquet(file)
    print(file)
    print(df.shape)
    print()

analytics/03-01-2018-analytics.parquet
(329193, 9)

analytics/02-23-2018-analytics.parquet
(1042207, 9)

analytics/02-16-2018-analytics.parquet
(900988, 9)

analytics/02-15-2018-analytics.parquet
(1041233, 9)

analytics/02-20-2018-analytics.parquet
(7911980, 9)

analytics/02-21-2018-analytics.parquet
(1031018, 9)

analytics/02-28-2018-analytics.parquet
(604313, 9)

analytics/03-02-2018-analytics.parquet
(1040558, 9)

analytics/02-14-2018-analytics.parquet
(820670, 9)

analytics/02-22-2018-analytics.parquet
(1041733, 9)



In [3]:
cleaned_files = glob("cleaned/*.parquet")

In [4]:
for file in cleaned_files:
    df = pd.read_parquet(file)
    print(file)
    print(df.shape)
    print()

cleaned/02-28-2018-clean.parquet
(604313, 80)

cleaned/03-01-2018-clean.parquet
(329193, 80)

cleaned/03-02-2018-clean.parquet
(1040558, 80)

cleaned/02-23-2018-clean.parquet
(1042207, 80)

cleaned/02-15-2018-clean.parquet
(1041233, 80)

cleaned/02-14-2018-clean.parquet
(820670, 80)

cleaned/02-21-2018-clean.parquet
(1031018, 80)

cleaned/02-16-2018-clean.parquet
(900988, 80)

cleaned/02-22-2018-clean.parquet
(1041733, 80)



Double check that all Duplicates, missing cells and such were removed. As well as timestamp type updated

In [5]:
for file in cleaned_files:
    df = pd.read_parquet(file)

    print(file)
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())
    print("Missing Flow Byts/s:", df["Flow Byts/s"].isna().sum())
    print("Bad Label rows:", (df["Label"] == "Label").sum())
    print("Missing Timestamp:", df["Timestamp"].isna().sum())
    print("Timestamp dtype:", df["Timestamp"].dtype)
    print()

cleaned/02-28-2018-clean.parquet
Shape: (604313, 80)
Duplicates: 0
Missing Flow Byts/s: 0
Bad Label rows: 0
Missing Timestamp: 0
Timestamp dtype: datetime64[us]

cleaned/03-01-2018-clean.parquet
Shape: (329193, 80)
Duplicates: 0
Missing Flow Byts/s: 0
Bad Label rows: 0
Missing Timestamp: 0
Timestamp dtype: datetime64[us]

cleaned/03-02-2018-clean.parquet
Shape: (1040558, 80)
Duplicates: 0
Missing Flow Byts/s: 0
Bad Label rows: 0
Missing Timestamp: 0
Timestamp dtype: datetime64[us]

cleaned/02-23-2018-clean.parquet
Shape: (1042207, 80)
Duplicates: 0
Missing Flow Byts/s: 0
Bad Label rows: 0
Missing Timestamp: 0
Timestamp dtype: datetime64[us]

cleaned/02-15-2018-clean.parquet
Shape: (1041233, 80)
Duplicates: 0
Missing Flow Byts/s: 0
Bad Label rows: 0
Missing Timestamp: 0
Timestamp dtype: datetime64[us]

cleaned/02-14-2018-clean.parquet
Shape: (820670, 80)
Duplicates: 0
Missing Flow Byts/s: 0
Bad Label rows: 0
Missing Timestamp: 0
Timestamp dtype: datetime64[us]

cleaned/02-21-2018-clean.

In [ ]:
pd.read_parquet("02-20-2018-clean.parquet").shape()